# Vérification locale d'un CSV avant ingestion

Ce notebook vérifie un fichier présent sur le poste avant de lancer le job Cloud Run. Il réutilise `CSV_SCHEMA` et `validate_csv_header` de `ingestion/main.py`, puis contrôle le nombre de colonnes, les champs obligatoires et les types. Il ne lance pas `main.py` et n'appelle ni Drive, ni Cloud Storage, ni BigQuery.

Sélectionne le kernel Python de l'environnement virtuel du projet, avec les dépendances de `ingestion/requirements.txt` installées.

In [ ]:
import csv
import io
import math
import sys
from datetime import date
from pathlib import Path

# Retrouve la racine même si le kernel a démarré depuis notebooks/.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'ingestion' / 'main.py').is_file():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError('Racine du dépôt introuvable depuis le dossier courant.')
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Importe le schéma et le contrôle d'en-tête réellement utilisés par l'ingestion.
from ingestion.main import CSV_SCHEMA, validate_csv_header

[(field.name, field.field_type, field.mode) for field in CSV_SCHEMA]

In [ ]:
# Modifie ces deux valeurs pour vérifier un autre fichier ou un CSV séparé par ;.
CSV_PATH = REPO_ROOT / 'sample_data' / 'market_prices.csv'
CSV_DELIMITER = ','
CSV_PATH

In [ ]:
def validate_local_csv(csv_path: Path, delimiter: str = ',') -> list[dict[str, object]]:
    """Valide un CSV local contre le schéma de l'ingestion, sans accès cloud."""
    csv_path = Path(csv_path)
    content = csv_path.read_bytes()

    # Réutilise le contrôle d'en-tête du programme d'ingestion.
    validate_csv_header(content, delimiter, csv_path.name)

    text = content.decode('utf-8-sig')
    reader = csv.reader(io.StringIO(text, newline=''), delimiter=delimiter, strict=True)
    next(reader, None)  # L'en-tête a déjà été validé ci-dessus.
    records: list[dict[str, object]] = []

    try:
        for row in reader:
            line_number = reader.line_num  # Numéro de ligne physique, y compris les champs cités.
            if not row:  # Ignore les lignes complètement vides.
                continue

            if len(row) != len(CSV_SCHEMA):
                raise ValueError(
                    f'{csv_path.name}, ligne {line_number} : {len(row)} colonne(s) '
                    f'au lieu de {len(CSV_SCHEMA)}.'
                )

            values = {field.name: value.strip() for field, value in zip(CSV_SCHEMA, row)}
            for field in CSV_SCHEMA:
                value = values[field.name]
                if value == '':
                    if field.mode == 'REQUIRED':
                        raise ValueError(
                            f'{csv_path.name}, ligne {line_number} : '
                            f"le champ obligatoire '{field.name}' est vide."
                        )
                    continue

                if field.field_type == 'DATE':
                    try:
                        parsed_date = date.fromisoformat(value)
                    except ValueError as exc:
                        raise ValueError(
                            f'{csv_path.name}, ligne {line_number} : '
                            f"date invalide dans '{field.name}' : {value!r}."
                        ) from exc
                    if parsed_date.isoformat() != value:
                        raise ValueError(
                            f'{csv_path.name}, ligne {line_number} : '
                            f"'{field.name}' doit être au format AAAA-MM-JJ."
                        )
                elif field.field_type == 'FLOAT':
                    try:
                        number = float(value)
                    except ValueError as exc:
                        raise ValueError(
                            f'{csv_path.name}, ligne {line_number} : '
                            f"valeur numérique invalide dans '{field.name}' : {value!r}."
                        ) from exc
                    if not math.isfinite(number):
                        raise ValueError(
                            f'{csv_path.name}, ligne {line_number} : '
                            f"valeur non finie dans '{field.name}' : {value!r}."
                        )
                elif field.field_type == 'INT64':
                    try:
                        int(value, 10)
                    except ValueError as exc:
                        raise ValueError(
                            f'{csv_path.name}, ligne {line_number} : '
                            f"'{field.name}' doit être un entier : {value!r}."
                        ) from exc

            records.append({'line_number': line_number, **values})
    except csv.Error as exc:
        raise ValueError(
            f'{csv_path.name}, CSV mal formé vers la ligne {reader.line_num} : {exc}'
        ) from exc

    if not records:
        raise ValueError(f'{csv_path.name} ne contient aucune ligne de données.')

    return records

In [ ]:
records = validate_local_csv(CSV_PATH, CSV_DELIMITER)
print(f'CSV valide : {len(records)} ligne(s) de données, {len(CSV_SCHEMA)} colonnes.')
records[:5]  # Aperçu sous forme de liste de dictionnaires Python.

## Ce que valide ce contrôle

- L'encodage UTF-8 (avec ou sans BOM) et l'en-tête attendu par `CSV_SCHEMA`.
- Le nombre de colonnes de chaque ligne et la présence des champs obligatoires.
- Les dates `AAAA-MM-JJ`, les nombres décimaux finis et les volumes entiers.

La validation s'effectue uniquement sur le fichier local. Elle ne teste ni les permissions Google Cloud, ni le chargement BigQuery lui-même. Après un résultat valide, le job d'ingestion complet peut être lancé séparément.